# FoBench Basics
Welcome to the tutorial of the basic functionalities of FoBench. This notebook will guide you through the loading of data, trimming it in space and time, filtering and some visuals. Of course we can not go through every method in detail, so feel free to dive into the [documentation](https://doctus5.github.io/fobench/) if you want to know more. Here and there we will hint to more functionality to discover but for now we will stick to the basics.

A note about plotting: 
Most methods have two plotting modes: matplotlib (`plot_mode="mpl"`) and PyQtGraph (`plot_mode="pyqt"`), the standard in FoBench is using PyQtGraph as it performs better when it comes to larger matrix plots. However it is not very stable when working in a Jupyter notebook, it should however work from other IDEs or simple scripts. If you want to know more on how to navigate FoBench plots, check out the page on documentation on [visualisations](https://doctus5.github.io/fobench/visualizations.html).


In [ ]:
import os
os.environ['PYQTGRAPH_QT_LIB'] = 'PyQt5'
from fobench import Fiber
%gui qt

The workhorse of FoBench is the `Fiber` class, it contains the actual fiber optic record with all its metadata and provides a lot of functionality to manipulate data in space and time. Let us go right ahead and load a file. We simply give the path to the file and we let FoBench know the manufacturer of the interrogator we used. In our case we will use data recorded on an Aragon Photonics HDAS system. You can find the supported data formats and corresponding keywords in the documentation. You will find all the formats that FoBench supports and the corresponding keywords [here](https://doctus5.github.io/fobench/getting_started/formats.html).

In [ ]:
das = Fiber('./example_data/aragon_h5/2024_07_11_05h15m16s_HDAS_2DRawData_Strain.h5', 'aragon')

Lets first have a look at the most important metadata features of the file we loaded.

In [ ]:
print(das)
# das.metadata() # for the full metadata

We can see that the file contains a single minute of strain data, lets concatenate a second file and convert into strain-rate after. 

In [ ]:
das += Fiber('./example_data/aragon_h5/2024_07_11_05h16m16s_HDAS_2DRawData_Strain.h5', 'aragon') # the += syntax just calls Fiber.concatenate()
das.differentiate() # Fiber.integrate for integration of the data

Our original sampling frequency is a bit too high, we will decimate the data to 50 Hz.

If we then check the aquisition parameters again, we can see that we now have two minutes of strain-rate data at 50 Hz

In [ ]:
new_freq = 50
das.decimate(new_freq)
print(das)

The `Fiber` class stores the actual data in the `Fiber.data` attribute. This way we can easily access and extract it:

In [ ]:
print(type(das.data), das.data.shape, das.data)
#das.get_data() # returns full data or data of a specific channel
#das.times # returns the time stamps for each sample in specified format

Most operations on the data are done inplace. Just in case we mess something up later on, at any point we can get a copy of the records current state with `Fiber.copy()`

In [ ]:
backup_das = das.copy()

Before we have a look at the data, we should apply some basic preprocessing. Let us start with a simple bandpass filter between 0.1 and 20 Hz. We can check if the filter was applied successfully by looking at the list stored in `Fiber.processing`:

In [ ]:
das.filter(f_type='bandpass', freq=(1, 15)) # other filter options are highpass, lowpass and bandstop. Fobench also provides more specialized filters such as median, Cheby and FIR-filters
das.processing

The list keeps track of all preprocessing steps included in FoBench so that at any point we can go back and check which steps we took in what order. This is done by storing the methods name and all parameters it is called with.

We can see that before bandpass filtering the data was preprocessed, in FoBench this mean demeaning, detrending and tapering. The filter function by default conveniently performs these steps before the actual filtering. Of course all these methods can be used individually and `filter` can be used with the `pre_process` parameter set to `False`.

Note also that most methods have additional parameters such as the removal of higher order polynomial trends by passing the `order` to `Fiber.detrend()`. A lot of methods can be applied in both space and time, this is determined by the `dim` parameter.

Now back to our data! Let us plot the record in both time and frequency domain:

In [ ]:
das.plot()
das.fx_plot()

It seems that we have recorded a small event! However it looks like towards the end of the cable we have no useful data anymore. We will trim the record and focus on the channels 20 to 280

In [ ]:
das.restrict_channels(20, 280)
das.n_channels # number of channels in the record
# print(das.channels_num) # all channel numbers

Now that we have indentified the channels that we are interested in, the next time when we are loading data, we can retrieve only the part of interest. This can decrease the loading time significantly. We do that by simply passing the channel range to the Fiber class: 
```
file = './example_data/2024_07_11_05h15m16s_HDAS_2DRawData_Strain.h5'
manufacturer = 'aragon'
channels = [20, 280]
das = Fiber(file, manufacturer, range_ch=channels)
```
We can also trim the record around the time of the event and plot again:

In [ ]:
t0 = das.start_time + 43
tf = t0 + 16
das.trim(t0, tf)
das.plot()

We can look at a few channels in more detail, lets go for 100 - 120:

In [ ]:
section = (100, 120)
das.record_section(section)

We can plot a single channels waveform and the corresponding spectrogram and amplitude spectrum:

In [ ]:
ch = 110
das.channel_plot(ch)

To explore the data a bit deeper we can call the data explorer:

In [ ]:
das.view()